# 08 · Risk Analysis & Competitive Gap Assessment
**Project:** PipelineIQ CRM Analytics | **Date:** March 2026

## Purpose
Two separate but connected analyses:

1. **Risk Analysis:** Quantify the downside scenarios the business faces — Monte Carlo MRR simulation, customer concentration risk (HHI), CAC payback cliff, and the NRR Treadmill showing how much new MRR is required to hit ARR targets at current NRR levels.

2. **Gap Analysis:** Benchmark eight key SaaS metrics against OpenView 2024, Bessemer 2024, and KeyBanc 2024 benchmarks. Classify each gap as Critical / Below Benchmark / At Benchmark and generate a prioritised action matrix.

> **Why these belong together:** Risk analysis tells you what can go wrong. Gap analysis tells you where the business already sits relative to the market. Together they answer: "What is the most important thing to fix, and how bad does it get if we don't?" 

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, matplotlib.patches as mpatches
import warnings; warnings.filterwarnings('ignore'); np.random.seed(42)

cust = pd.read_csv('clean_saas_customers.csv')
cust['mrr'] = pd.to_numeric(cust['mrr'],errors='coerce').clip(5,500)
cust['tier'] = cust['tier'].astype(str).str.lower().str.strip()
current_mrr = cust['mrr'].sum()
print(f"Current total MRR: ${current_mrr:,.0f}")
print(f"Annualised ARR:    ${current_mrr*12:,.0f}")

## Section 1: Monte Carlo MRR Simulation
We simulate 5,000 paths of 12-month MRR evolution, drawing monthly churn rate, expansion rate, and new MRR from distributions calibrated to the last 6 months of observed data. The fan chart shows P5/P25/P50/P75/P95 outcomes.

**Calibration note:** `churn_mean=0.0718` is the customer-months monthly churn rate from the actual dataset. It is NOT `churned.mean()` (which gives 68.2% and is the lifetime churn rate, not a monthly rate). Using the wrong number here would produce nonsensical MRR projections — the simulation would zero out MRR within 3 months.

In [ ]:
N_SIM=5000; MONTHS=12
# Calibrated parameters from observed data
churn_mean=0.0718; churn_std=0.018  # calibrated from actual data: 7.18%/mo overall, higher Starter variance
expansion_mean=0.018; expansion_std=0.008
new_mrr_mean=1200; new_mrr_std=400

paths = np.zeros((N_SIM, MONTHS+1))
paths[:,0] = current_mrr
for sim in range(N_SIM):
    mrr = current_mrr
    for mo in range(MONTHS):
        ch  = np.random.normal(churn_mean, churn_std)
        exp = np.random.normal(expansion_mean, expansion_std)
        new = np.random.normal(new_mrr_mean, new_mrr_std)
        mrr = mrr*(1 - max(0,ch) + max(0,exp)) + max(0,new)
        paths[sim, mo+1] = mrr

pcts = {5:np.percentile(paths,5,axis=0), 25:np.percentile(paths,25,axis=0),
        50:np.percentile(paths,50,axis=0), 75:np.percentile(paths,75,axis=0),
        95:np.percentile(paths,95,axis=0)}
mo_axis = range(MONTHS+1)
fig, ax = plt.subplots(figsize=(12,6))
ax.fill_between(mo_axis, pcts[5],  pcts[95], alpha=0.10, color='#2980b9', label='P5–P95')
ax.fill_between(mo_axis, pcts[25], pcts[75], alpha=0.20, color='#2980b9', label='P25–P75')
ax.plot(mo_axis, pcts[50], color='#2980b9', linewidth=3, label='P50 (median)')
ax.plot(mo_axis, pcts[5],  color='#e74c3c', linewidth=1.5, linestyle='--', label='P5 (bear)')
ax.plot(mo_axis, pcts[95], color='#27ae60', linewidth=1.5, linestyle='--', label='P95 (bull)')
ax.set_xlabel('Months forward'); ax.set_ylabel('MRR ($)')
ax.set_title('Monte Carlo MRR Simulation — 5,000 Paths (12-Month Forward)', fontweight='bold')
ax.legend(); plt.tight_layout()
plt.savefig('../reports/risk_monte_carlo.png',dpi=150) if __import__('os').path.exists('../reports') else None
plt.show()
print(f"P50 12-month MRR: ${pcts[50][-1]:,.0f}")
print(f"P5  12-month MRR: ${pcts[5][-1]:,.0f} (bear case)")
print(f"P95 12-month MRR: ${pcts[95][-1]:,.0f} (bull case)")
print(f"Probability of MRR decline: {(paths[:,-1]<current_mrr).mean():.1%}")

## Section 2: Customer Concentration Risk (HHI)
The Herfindahl-Hirschman Index measures revenue concentration. HHI = Σ(sᵢ²) where sᵢ is each customer's MRR share. HHI > 0.25 signals dangerous concentration — single-customer dependency.

In [ ]:
cust['mrr_share'] = cust['mrr'] / cust['mrr'].sum()
hhi = (cust['mrr_share']**2).sum()
top1  = cust['mrr_share'].nlargest(1).sum()
top5  = cust['mrr_share'].nlargest(5).sum()
top10 = cust['mrr_share'].nlargest(10).sum()
print(f"=== CONCENTRATION RISK ===")
print(f"  HHI:            {hhi:.6f} ({'LOW — diversified' if hhi<0.01 else 'MODERATE' if hhi<0.25 else 'HIGH — dangerous concentration'})")
print(f"  Top 1 customer: {top1:.1%} of MRR")
print(f"  Top 5 customers:{top5:.1%} of MRR")
print(f"  Top 10 customers:{top10:.1%} of MRR")
print(f"\n  Benchmark: Healthy SaaS should have top-10 <20% MRR concentration")
print(f"  Status: {'✓ Within bounds' if top10<0.20 else '⚠ Concentration risk — largest account loss would be material'}")

## Section 3: NRR Treadmill
Net Revenue Retention (NRR) determines how much new MRR is required each month just to maintain ARR growth targets. Lower NRR means the business must run faster on new acquisition just to stand still.

In [ ]:
target_arr_growth = 0.50  # 50% ARR growth target
nrr_range = np.arange(0.70, 1.21, 0.05)
arr_now = current_mrr * 12
required_new_mrr = []
for nrr in nrr_range:
    # ARR next year = ARR_now × NRR + New_ARR
    # Required New_ARR = ARR_now × (1+growth) - ARR_now × NRR
    req_new_arr = arr_now * (1 + target_arr_growth) - arr_now * nrr
    req_new_mrr = max(0, req_new_arr / 12)
    required_new_mrr.append(req_new_mrr)

fig, ax = plt.subplots(figsize=(10,5))
colors = ['#e74c3c' if r>3000 else '#f39c12' if r>1500 else '#27ae60' for r in required_new_mrr]
bars = ax.bar([f"{n:.0%}" for n in nrr_range], required_new_mrr, color=colors, edgecolor='white', linewidth=0.5)
ax.axhline(y=new_mrr_mean, color='black', linestyle='--', alpha=0.6, label=f'Current new MRR capacity (~${new_mrr_mean:,}/mo)')
ax.set_xlabel('Net Revenue Retention (NRR)'); ax.set_ylabel('Required New MRR/month ($)')
ax.set_title(f'NRR Treadmill — Required New MRR to Hit {target_arr_growth:.0%} ARR Growth', fontweight='bold')
ax.legend(); plt.tight_layout()
plt.savefig('../reports/risk_nrr_treadmill.png',dpi=150) if __import__('os').path.exists('../reports') else None
plt.show()
estimated_nrr = 0.85  # estimated from data
req = [r for n,r in zip(nrr_range, required_new_mrr) if abs(n-estimated_nrr)<0.03]
print(f"At estimated NRR={estimated_nrr:.0%}: requires ${req[0] if req else 0:,.0f}/mo new MRR for {target_arr_growth:.0%} ARR growth")

## Section 4: Gap Analysis vs Industry Benchmarks
Benchmarks sourced from: OpenView 2024 SaaS Benchmarks (n=600+ private SaaS, ACV $5k-50k), Bessemer Venture Partners 2024 State of the Cloud, KeyBanc 2024 SaaS Survey.

In [ ]:
gaps = pd.DataFrame([
    {'metric':'Net Revenue Retention','ours':85.0,'p25':95.0,'median':106.0,'p75':118.0,'source':'OpenView 2024','unit':'%'},
    {'metric':'CAC Payback (Starter)','ours':12.2,'p25':18.0,'median':14.0,'p75':10.0,'source':'KeyBanc 2024','unit':'months','lower_better':True},
    {'metric':'Gross Margin','ours':72.0,'p25':68.0,'median':74.0,'p75':80.0,'source':'Bessemer 2024','unit':'%'},
    {'metric':'Churn Rate (monthly)','ours':7.18,'p25':3.0,'median':2.0,'p75':1.5,'source':'OpenView 2024','unit':'%','lower_better':True},
    {'metric':'Logo Churn (annual)','ours':59.1,  # 1-(1-0.0718)^12'p25':15.0,'median':10.0,'p75':6.0,'source':'KeyBanc 2024','unit':'%','lower_better':True},
    {'metric':'Magic Number','ours':0.42,'p25':0.50,'median':0.75,'p75':1.10,'source':'Bessemer 2024','unit':'x'},
    {'metric':'ARR/FTE','ours':95.0,'p25':80.0,'median':120.0,'p75':180.0,'source':'OpenView 2024','unit':'$k'},
    {'metric':'LTV:CAC Ratio','ours':2.1,'p25':3.0,'median':5.0,'p75':8.0,'source':'KeyBanc 2024','unit':'x'},
])
def classify(row):
    lb = row.get('lower_better', False)
    v, p25, med = row['ours'], row['p25'], row['median']
    if lb: return 'Critical' if v>p25*1.5 else 'Below Benchmark' if v>med else 'At Benchmark'
    else:  return 'Critical' if v<p25*0.7 else 'Below Benchmark' if v<med else 'At Benchmark'
gaps['status'] = gaps.apply(classify, axis=1)
gaps['priority'] = gaps['status'].map({'Critical':'P1','Below Benchmark':'P2','At Benchmark':'P3'})
print("=== GAP ANALYSIS vs INDUSTRY BENCHMARKS ===")
print(gaps[['metric','ours','p25','median','p75','status','priority','source']].to_string(index=False))
critical = gaps[gaps['status']=='Critical']['metric'].tolist()
print(f"\nCritical gaps (P1): {critical}")

## Prioritised Action Matrix

Ordered by: (1) financial impact, (2) feasibility, (3) time to effect.

In [ ]:
actions = pd.DataFrame([
    {'priority':'P1','metric':'NRR (85% vs 106% median)','action':'Annual billing conversion campaign (Experiment 5) + CSM At Risk outreach (RFM module)',
     'expected_impact':'NRR +8-15pp in 6 months','timeline':'30 days to launch','owner':'CS + Product'},
    {'priority':'P1','metric':'Logo churn (55% vs 10% median)','action':'Starter 30-60-90 onboarding redesign (Experiment 3) + pricing review',
     'expected_impact':'Logo churn -15-20pp in 12 months','timeline':'60 days to build','owner':'Product + CS'},
    {'priority':'P1','metric':'LTV:CAC (2.1x vs 5.0x median)','action':'Shift acquisition spend from Paid Search to Referral; add formal referral programme',
     'expected_impact':'LTV:CAC +1.5-2.5x in 12 months','timeline':'45 days','owner':'Marketing'},
    {'priority':'P1','metric':'Monthly churn (6.8% vs 2.0% median)','action':'RFM-triggered CSM alerts + product health scoring dashboard',
     'expected_impact':'Churn -2-3pp in 6 months','timeline':'30 days','owner':'CS'},
    {'priority':'P2','metric':'Magic Number (0.42 vs 0.75 median)','action':'Pro repricing experiment ($79→$40) to improve sales efficiency',
     'expected_impact':'+0.2-0.3 improvement in 6 months','timeline':'30 days to test','owner':'Finance + Product'},
    {'priority':'P2','metric':'ARR/FTE ($95k vs $120k median)','action':'Automate Starter onboarding; reduce CS headcount dependency',
     'expected_impact':'+$15-25k ARR/FTE in 12 months','timeline':'90 days','owner':'Engineering'},
])
print(actions[['priority','metric','action','expected_impact','timeline','owner']].to_string(index=False))